# Pendulum Dynamics 2: Finding a Transfer Function Based on Pulse and Step Inputs

## Overview

In the previous lab you investigated whether or not a pendulum can be modeled as a mass/spring/damper system based on an initial condition input and tapping the pendulum with a pencil or finger.  In this lab, you will continue working with cart/pendulum system.  This week you will use voltage to the cart motors as the input and try to find a transfer function for the system with the encoder measurement as the output:

$$G(s) = \frac{Out(s)}{In(s)} = \frac{Encoder(s)}{Voltage(s)}$$

This will be your first attempt to find this transfer function.  We will also attempt to find the transfer function using a Bode plot in a future lab.  In this lab, you will use pulse inputs and a step input.

## Block Diagram Creation

The plant for this lab will be the same as last week:

- `plant_with_double_actuator`
    - custom actuator with Arduino class `cartmotors`
    - the actuator name cannot be empty and must be a valid `c` variable name
         - `myact` would be fine
    - the sensor is an encoder with sensitivity 4

<img src="figs/myact_cartmotors.png" width=300px>


### You *might* be able to reuse last week's block diagram

If you are lucky and my `wxbd_gui` software works correctly, you might be able to replace the constant zero block with a `pulse_input`.  If replacing the input block doesn't work, it shouldn't take very long to create a new model from scratch. 

### Pulse Input

For the first part of this lab, you will give the cart motors a short pulse.  It will not be a true impulse because the amplitude is limited to $\pm$400 counts.  Try an amplitude of 300 and an off time of 0.3 seconds as a starting point.

### Block Diagram for Pulse Input

The block diagram for the pulse input part of the lab is shown below:

<img src="figs/lab_4_pulse_bd.png" width=600px>

## Pulse Input Testing

It should be fairly straightforward to run a pulse test with the generated code.  The cart should move forward a small amount and the pendulum should swing.  If those things don't happen, you will need to debug.  

Assuming the cart moves and the pendulum swings, copy and paste the data from the serial monitor to a text or csv file like last week. 

## Two Different Transfer Functions

You will attempt to fit the pulse and step response data using two different transfer functions.  The first will be based on an MSD system with force input and displacement output:

<img src="figs/forced_damped_vibration.png" width=300px>

**Figure 1:** A mass/spring/damper system with force input and displacement output.

$$G_1(s) = \frac{X(s)}{F(s)}$$

The second transfer function is based on modeling a pendulum as a displacement input:

$$G_2(s) = \frac{X(s)}{U(s)}$$

You will investigate which of these two transfer functions fits the system better based on curve fitting the pulse and step responses.

<img src="figs/Pendulum_approx_msd_v2_cropped.png" width=600px>

**Figure 2:** Modeling a pendulum as a mass/spring/damper system with a displacement input.

## Pulse Input Curve Fitting

Once you have derived the two transfer functions $G_1$ and $G_2$, you are ready to use them to curve fit your pulse response data.  The curve fitting process will be based on `scipy.optimize.fmin` as in previous labs.  See the helper notebook from lab 2 for more details: [fmin_intro_example_with_loadtxt_error_examples_v2.ipynb](https://github.com/kraussry/student_git_345_F26/blob/main/lab_345/lab_02_intro_to_wxbd_and_RC_step_response/helper_notebooks/fmin_intro_example_with_loadtxt_error_examples_v2.ipynb)

When using `optimze.fmin`, you will always need a cost function that returns a scalar value that is the sum of the squared errors:

In [9]:
def mycost1(c):
    y_model = mymodel1(c)
    error_vect = y-y_model
    e_sum = np.sum(error_vect**2)
    return e_sum

You will also need a `mymodel1` function that returns the system response that you are trying to fit to the data.  For this lab, you will want to use `control.forced_resonse` inside of `mymodel1` to find the transfer function response to the experimental input:

In [7]:
def mymodel1(c):
    # unpack parameters here
    a = c[0]
    # probably more parameters here

    # create the transfer function based on your parameters
    # define num and den here
    G1 = TF(num,den)

    # use control.forced_response to find the model response
    to, y_model = control.forced_response(G,fake_t,u)#u would be the input 
                                                #to the cart motors
    return y_model

**Note:** You would need to create `mycost` and `mymodel2` functions that use $G_2(s)$.

## Step Input Testing

The only difference between the pulse input testing and the step input testing is the voltage sent to the motors.  You might be able to replace the pulse input with a step input if things go well.  A couple of things to note about the step input testing:

- you will need an amplitude of around 300 to over come friction and get the cart to move
- the cart is going to take off and keep going for the length of the test
    - look for the variable `stop_t` in the Arduino code or template and change it to 3
    - plan on the cart moving 4-5 feet during the test
        - use a long USB cable

## Step Input Curve Fitting

There will likely be a qualitative difference between the experimental pendulum data and the model response.  The main point of the step response testing is to help you realize that the current models are imperfect.

## Comprehension Question

What was qualitatively different between the step responses of the models and the experimental data?  How would you explain this qualitative issue conceptually?  What is it that is different between the physical system and the two models you are using in this lab?